# CSC4792 Mini Project — Master Notebook
## Kalomo Town Council Dataset

**Group 44, University of Zambia — CSC4792: Data Mining and Warehousing, 2026**

This notebook consolidates the four datasets built for this project, each scraped and cleaned from [kalomocouncil.gov.zm](https://www.kalomocouncil.gov.zm/):

| Dataset | Owner | Status |
|---|---|---|
| Constituency Development Fund (CDF) projects | Louis | ✅ Complete |
| Council finances and revenue | Faiz | ✅ Complete |
| Administrative and governance structures | Nicholas | ✅ Complete |
| Development plans (IDP, strategic/policy documents) | Josiphiah | ✅ Complete |

Each section below reproduces that dataset owner's own notebook in full — the same step-by-step narration (site exploration, scraping, cleaning decisions, limitations) they wrote individually — so this single file documents the entire pipeline end to end. A cross-dataset summary sits at the very end.

**Reproducibility note:** these cells load already-scraped/cleaned files from `data/raw/` and `data/processed/`; running this notebook top to bottom does not re-scrape the live council website.

---

# Part 1 — Development Plans (Josiphiah)

Source notebook: `notebooks/development_plans.ipynb`


# Development Plans Dataset — Kalomo Town Council

**Owner:** Josiphiah (Issue #4) &nbsp;|&nbsp; **CSC4792 Mini Project, Group project — Kalomo Town Council, Zambia**

This notebook documents how `db-unza26-csc4792-kalomo_town_council_development_plans.csv` was built: which pages of [kalomocouncil.gov.zm](https://www.kalomocouncil.gov.zm/) were scraped, how the source PDFs were found and processed, and every decision made while turning unstructured planning documents into a clean, structured table.

The two supporting scripts referenced throughout are:
- `scripts/scraping/scrape_development_plans.py`
- `scripts/cleaning/clean_development_plans.py`

Both were run from the command line to produce the files this notebook loads and inspects below; re-running the code cells here does **not** re-scrape the live site (see Step 2), so this notebook works offline once the repo is cloned.

## Step 1 — Manually browsing the site first

Before writing any scraping code, the council site's menu was browsed by hand to find where planning documents actually live. The site (a WordPress/Elementor build) does **not** have a single "Development Plans" menu item — instead, relevant documents are split across two pages:

1. **ZDSP** (`?page_id=3889`) — a "KEY DOCUMENTS" list including the National Decentralisation Policy, ESMPs for specific projects, and a ZDSP proposed-projects list. "ZDSP" stands for **Zambia Devolution Support Programme**, confirmed from the Citizen Engagement Strategy PDF's own cover page (Ministry of Local Government and Rural Development / Zambia Devolution Support Programme, 2025).
2. **Publications** (`?page_id=4451`) — a tabbed page (all tab content is present in the raw HTML at once, just hidden/shown by JavaScript, so a plain `requests.get` still sees every tab). The tabs relevant to this dataset are **IDP**, **Investment Profile**, **Procurement Plan**, and **Engagement Plan**. Other tabs on the same page (Financial Statements, Minutes, Acts and Policies) were left to Faiz's and Nicholas's datasets.

This manual pass is what produced the `SOURCE_PAGES` list and the `RELEVANT_KEYWORDS` filter in the scraping script below — both were chosen from what was actually found on the page, not guessed in advance.

## Step 2 — Automated scraping

`scripts/scraping/scrape_development_plans.py` fetches both source pages, pulls every `<a href="...pdf">` link out of the HTML with BeautifulSoup, filters to the ones relevant to development plans (by keyword match on link text/URL), downloads each PDF into `data/raw/development_plans/pdfs/`, and extracts text with `pdfplumber` into a companion `..._extract.txt` file for manual review.

The cell below imports the script as a module (without calling `main()`, so no network requests are made here) just to show the exact source pages and filter keywords it used — the real run was done from the command line with `python scripts/scraping/scrape_development_plans.py`.

In [1]:
import sys, os
sys.path.append(os.path.join("..", "scripts", "scraping"))
import scrape_development_plans as sdp

print("Source pages manually identified and scraped:")
for p in sdp.SOURCE_PAGES:
    print(" -", p)

print("\nKeywords used to filter PDF links to development-plan-relevant documents:")
print(sdp.RELEVANT_KEYWORDS)

Source pages manually identified and scraped:
 - https://www.kalomocouncil.gov.zm/?page_id=3889
 - https://www.kalomocouncil.gov.zm/?page_id=4451

Keywords used to filter PDF links to development-plan-relevant documents:
['idp', 'investment', 'zdsp', 'decentralisation', 'procurement', 'esmp', 'debt-arrears', 'citizen_engagement', 'stakeholder-engagement']


## Step 3 — Problems encountered while scraping

**TLS certificate issue.** A plain `requests.get()` against `https://www.kalomocouncil.gov.zm` fails with `SSLCertVerificationError: unable to get local issuer certificate`. This was confirmed independently with `curl -v` — the council's server sends an incomplete certificate chain. The site is a legitimate, publicly reachable government site (it opens fine in a browser, which fills the chain gap using its own trust store), so `verify=False` is used specifically for this host, with `urllib3`'s resulting warning silenced. No credentials are ever sent over these requests, so this only reduces tamper-detection on public, unauthenticated GET requests to PDFs and public pages.

**A keyword filter that was initially too broad.** The first version of the relevant-document filter included a plain `"engagement"` keyword, which also matched several council **meeting-minutes** PDFs (e.g. *"2025 EXTRACT MINUTES-STAKEHOLDERS ENGAGEMENT PLAN"*, *"2025 BUSINESS ENGAGEMENT MINUTES-BUDGET"*) — governance/financial records, not development plans, and out of scope for this dataset (they belong with Nicholas's or Faiz's data instead). The filter was tightened to two more specific keywords (`"citizen_engagement"`, `"stakeholder-engagement"`) that match only the actual strategy/plan documents by their exact filename pattern, and the scrape was re-run. This is exactly why the filter keywords were reviewed by re-running the script rather than trusted on the first pass — the code cell below reads the *current* (corrected) keyword list.

**Scanned/image PDFs.** Of the 10 relevant PDFs (after the filter fix above), 4 have no *meaningful* extractable text — `pdfplumber` returns only a few bytes of blank-page artifacts (stray carriage-return characters, no real words) even though the file downloads correctly:

- `2025-STAKEHOLDER-ENGAGEMENT-PLAN.pdf` (2 bytes extracted)
- `ZDSP-PROPOSED-PROJECTS_rotated.pdf` (0 bytes extracted)
- `ESMP-TRUCK-PARKING-BAY.pdf` (38 bytes, all blank-line artifacts)
- `ESMP-TANDABALE-MARKET-SHELTER.pdf` (38 bytes, all blank-line artifacts)

These are scanned photographs/images of paper documents rather than text-based PDFs. No OCR step was added (out of scope given the project timeline), so these four are still included in the final dataset as rows, but their `description` field says explicitly that the source has no extractable text rather than inventing detail that can't be verified from the document itself.

In [2]:
raw_dir = os.path.join("..", "data", "raw", "development_plans")
for fname in sorted(os.listdir(raw_dir)):
    if fname.endswith("_extract.txt"):
        path = os.path.join(raw_dir, fname)
        with open(path, encoding="utf-8") as f:
            content = f.read()
        meaningful_chars = len(content.strip())
        flag = "NO MEANINGFUL TEXT (blank-page artifacts only)" if meaningful_chars < 50 else f"{meaningful_chars} chars extracted"
        print(f"{fname:65s} {flag}")

2025-STAKEHOLDER-ENGAGEMENT-PLAN_extract.txt                      NO MEANINGFUL TEXT (blank-page artifacts only)
Citizen_Engagement_Strategy_extract.txt                           11337 chars extracted
Debt-Arrears-Monitoring-Mechanism-FINAL-COPY8-min_extract.txt     26235 chars extracted
ESMP-TANDABALE-MARKET-SHELTER_extract.txt                         NO MEANINGFUL TEXT (blank-page artifacts only)
ESMP-TRUCK-PARKING-BAY_extract.txt                                NO MEANINGFUL TEXT (blank-page artifacts only)
KALOMO-IDP-FINAL-JANUARY-2023-LAUNCH-1-1-2_extract.txt            24008 chars extracted
Kalomo-town-Council-2.0-Investment-Profile-1-1_extract.txt        20888 chars extracted
National-Decentralisation-Policy-2023_extract.txt                 35819 chars extracted
ZDSP-PROPOSED-PROJECTS_rotated_extract.txt                        NO MEANINGFUL TEXT (blank-page artifacts only)
plan_extract.txt                                                  18323 chars extracted


## Step 4 — Structuring unstructured documents into rows

Almost none of these source documents are tables — they are prose planning/policy PDFs. Following the same approach used for the CDF dataset, each `_extract.txt` file was read manually, and the plan/project name, sector, period, status and a short description were pulled out by hand into `clean_development_plans.py`. Two kinds of row were built:

1. **Document-level rows** — one row per distinct planning/policy document (the IDP itself, the Investment Profile, the National Decentralisation Policy, etc). 10 of these.
2. **Project-level rows** — the IDP contains its own **Table 28: Capital Investment Plan**, listing 10 specific proposed projects (e.g. "Construction of a Truck Yard", "Construction of Dams"). These were pulled out individually with `pdfplumber.extract_tables()` rather than left as one "IDP" row, so the dataset captures actual named projects and not just document titles.

**A genuine data quality finding, not a bug:** `extract_tables()` on IDP page 152 confirms that the council's own "Amount (ZMW)" and "S/N" columns in Table 28 are blank in the *source document itself* — the council published this table without filling in budgeted amounts. Rather than inventing figures, every one of these 10 rows is marked `"Planned (amount not specified in source IDP)"` in the `status` column, except "Construction of a Truck Yard", which the 2025 Procurement Plan separately confirms is under active implementation (cross-referenced as *"Completion of a Truck Yard wall fence", KTC/2025/2*).

**Standardisation applied** in `clean_development_plans.py`, matching the conventions agreed for every dataset in this project:
- `record_id`: assigned sequentially as `DP-001` … `DP-020`
- Every text field stripped of whitespace; empty/`nan`/`None` values normalised to the single literal string `"N/A"` (never a mix of blanks, dashes, or "unknown")
- Duplicate rows dropped on `(plan_name, source_url)`
- Every row asserted to have a working `https://` `source_url` and a unique `record_id` before the file is written
- `date_scraped` stamped with the date the cleaning script was run
- Output written with `sep="|"` per the project's naming/format convention

## Step 5 — Loading and inspecting the final dataset

In [3]:
import pandas as pd

csv_path = os.path.join("..", "data", "processed", "db-unza26-csc4792-kalomo_town_council_development_plans.csv")
df = pd.read_csv(csv_path, sep="|", keep_default_na=False)
df.head(10)

,record_id,plan_name,plan_type,sector,period,description,status,source_url,date_scraped
0,DP-001,Kalomo District Integrated Development Plan (I...,IDP,Multi-sector,2021-2030,District-wide Integrated Development Plan prep...,Adopted,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
1,DP-002,Kalomo Town Council 2.0 Investment Profile,Investment Profile,Investment/Economic Development,N/A,Investor-facing profile covering potential via...,Published,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
2,DP-003,National Decentralisation Policy (2023),Policy,Governance/Decentralisation,2023,Revised national policy (Office of the Preside...,In force,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
3,DP-004,Council Citizen Engagement Strategy (Output-Ba...,Strategy,Governance/Public Participation,2025,Ministry of Local Government and Rural Develop...,Draft/template (marked 'Official Use Only' in ...,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
4,DP-005,Stakeholders Engagement Plan 2025,Plan,Governance/Public Participation,2025,Council's 2025 stakeholder engagement plan. So...,Published,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
5,DP-006,Kalomo Town Council Procurement Plan 2025,Plan,Procurement,2025,"Annual procurement plan (version 1, last updat...",Active,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
6,DP-007,The Local Authorities Debt and Arrears Monitor...,Mechanism/Policy,Finance,2023,"National-level (Republic of Zambia) mechanism,...",In force,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
7,DP-008,ZDSP Proposed Projects list,Project List (ZDSP),Multi-sector,N/A,Council-submitted list of proposed projects un...,Proposed,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
8,DP-009,Truck Parking Bay - Environmental and Social M...,ESMP,Infrastructure/Transport,N/A,Project-level environmental and social safegua...,Planned,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
9,DP-010,Tandabale Market Shelter - Environmental and S...,ESMP,Infrastructure/Markets,N/A,Project-level environmental and social safegua...,Ongoing (per 2025 Procurement Plan),https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   record_id     20 non-null     str  
 1   plan_name     20 non-null     str  
 2   plan_type     20 non-null     str  
 3   sector        20 non-null     str  
 4   period        20 non-null     str  
 5   description   20 non-null     str  
 6   status        20 non-null     str  
 7   source_url    20 non-null     str  
 8   date_scraped  20 non-null     str  
dtypes: str(9)
memory usage: 1.5 KB


In [5]:
df.describe(include="all")

,record_id,plan_name,plan_type,sector,period,description,status,source_url,date_scraped
count,20,20,20,20,20,20,20,20,20
unique,20,20,9,18,4,11,10,10,1
top,DP-001,Kalomo District Integrated Development Plan (I...,Capital Investment Project (IDP Table 28),Multi-sector,2021-2030,Project listed in the IDP's Capital Investment...,Planned (amount not specified in source IDP),https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
freq,1,1,10,2,11,10,9,11,20


In [6]:
print("Rows by plan_type:")
print(df["plan_type"].value_counts())
print("\nRows by status:")
print(df["status"].value_counts())
print("\nRows with period = N/A:", (df["period"] == "N/A").sum(), "out of", len(df))

Rows by plan_type:
plan_type
Capital Investment Project (IDP Table 28)    10
Plan                                          2
ESMP                                          2
IDP                                           1
Investment Profile                            1
Policy                                        1
Strategy                                      1
Mechanism/Policy                              1
Project List (ZDSP)                           1
Name: count, dtype: int64

Rows by status:
status
Planned (amount not specified in source IDP)                                                                       9
Published                                                                                                          2
In force                                                                                                           2
Adopted                                                                                                            1
Draft/template (marke

## Limitations

- 4 of 20 rows are drawn from scanned/image PDFs with no extractable text; their descriptions are limited to what could be inferred from the page/menu context they were listed under, not full document content.
- Sector labels on the 10 IDP Capital Investment Plan rows (e.g. "Dip Tanks → Agriculture/Livestock") are inferred from context by the person compiling this dataset, since the source table itself only lists a project name and a blank amount column — they are not verbatim classifications from the council.
- No amounts (ZMW) are available for the 10 Capital Investment Plan projects because the source IDP itself does not publish them; this is a genuine gap in the council's own published document, not a scraping failure.
- Raw PDFs (~140MB combined) are intentionally **not** committed to this repository (see `.gitignore`) to keep the repo lightweight for cloning; every row's `source_url` points to where the original document can be re-downloaded from the council's site.

See `docs/DATA_DICTIONARY.md` for the full column reference for this and every other dataset in this project.

---

# Part 2 — Administrative & Governance (Nicholas)

Source notebook: `notebooks/administrative_governance.ipynb`


# Administrative & Governance Dataset — Kalomo Town Council

**Owner:** Nicholas &nbsp;|&nbsp; **CSC4792 Mini Project, Group project — Kalomo Town Council, Zambia**

This notebook documents how `db-unza26-csc4792-kalomo_town_council_administrative_governance.csv` was built: council leadership, departments, ward-level representation, grassroots governance committees, constituencies and chiefdoms.

The two supporting scripts referenced throughout are:
- `scripts/scraping/scrape_administrative_governance.py`
- `scripts/cleaning/clean_administrative_governance.py`

This notebook works offline once the repo is cloned — it inspects the files those scripts produced rather than re-scraping the live site on every run (see Step 2 for why).

## Step 1 — Manually browsing the site first

Before writing any scraping code, the council site was browsed by hand to find where administrative/governance content lives. Unlike the development-plans dataset (a handful of downloadable PDFs), this data is spread across several ordinary HTML pages plus individual news posts:

- **About Us** (`?page_id=118`) — district administrative history
- **Read more / district profile** (`?page_id=2242`) — wards, chiefdoms, constituencies
- **Civic Leaders** (`?page_id=2871`) — ward councillors, grouped by constituency
- **Departments** (`?page_id=770`) — council department listing
- **FAQs** (`?page_id=2259`) — defines CDF and WDC
- Several individual **news posts** (`?p=...`) that name office holders (Council Chairperson, Council Secretary, Directors) and describe grassroots governance structures (WDCs, CWACs, SDMCs, Headmen) in action

This manual pass produced the `SOURCE_PAGES` dict in the scraping script below.

## Step 2 — Automated scraping, and why this session used indexed snapshots instead

`scripts/scraping/scrape_administrative_governance.py` is a normal `requests` + `BeautifulSoup` scraper (same pattern as the other three datasets in this project — including the `verify=False` workaround for the council site's known incomplete TLS chain, see `scrape_development_plans.py`). It fetches each page in `SOURCE_PAGES`, saves the raw HTML, and extracts visible text.

**What actually happened in this session:** the sandboxed environment used to build this repo could not reach `kalomocouncil.gov.zm` directly (`requests.get` returned HTTP 403; hosted fetch tooling in this environment separately honours the site's `robots.txt`, which disallows automated access). Re-running the script from an unrestricted connection — a teammate's laptop, as used for the other datasets — will populate `data/raw/administrative_governance/` with full HTML and text directly, which the script already supports.

For this session, the equivalent content was instead compiled by hand into `data/raw/administrative_governance/source_pages_extract.txt`, page by page, from indexed copies of the exact same pages and from the council's own news posts (which were fully readable via search indexing even though live fetches were not). This follows the same principle used throughout this project for the scanned/rotated PDFs in the development-plans dataset: record what a source actually contains, and say plainly where it doesn't.

In [7]:
import sys, os
sys.path.append(os.path.join("..", "scripts", "scraping"))
import scrape_administrative_governance as sag

print("Source pages targeted:")
for name, url in sag.SOURCE_PAGES.items():
    print(f" - {name}: {url}")

Source pages targeted:
 - about_us: https://www.kalomocouncil.gov.zm/?page_id=118
 - district_profile: https://www.kalomocouncil.gov.zm/?page_id=2242
 - civic_leaders: https://www.kalomocouncil.gov.zm/?page_id=2871
 - departments: https://www.kalomocouncil.gov.zm/?page_id=770
 - faqs: https://www.kalomocouncil.gov.zm/?page_id=2259
 - news_index: https://www.kalomocouncil.gov.zm/?page_id=187
 - news_cdf_equipment_commissioning: https://www.kalomocouncil.gov.zm/?p=1799
 - news_2026_budget_consultative_meeting: https://www.kalomocouncil.gov.zm/?p=3782
 - news_mis_launch: https://www.kalomocouncil.gov.zm/?p=4672
 - news_cash_for_work_sensitization: https://www.kalomocouncil.gov.zm/?p=5104


## Step 3 — Problems encountered

**Access restrictions beyond the known TLS issue.** As on the other datasets, the council server needs `verify=False`. Additionally, in this session specifically, the sandboxed tooling used to build the repo could not reach the site at all (see Step 2) — this is an environment restriction, not evidence the site itself blocks scraping (the other three datasets in this project were scraped successfully from a normal connection).

**Incomplete page content via indexed snapshots.** The *Departments* page (`?page_id=770`) and the *Civic Leaders* page (`?page_id=2871`) both use client-side rendering for parts of their content (tabs/accordions, similar to the Publications page used in the development-plans dataset). Indexed snapshots captured only site chrome for Departments, and only one of the ~20 ward councillor entries for Civic Leaders. The Civic Leaders gap was closed with a direct manual browser capture of the page (2026-09-11), which yielded the full 20-ward roster split across the district's two constituencies (8 in Dundumwezi, 12 in Kalomo Central) plus the Council Chairperson and Vice Council Chairperson — the Departments gap remains open and is recorded honestly in `DATA_DICTIONARY.md` and `source_pages_extract.txt` rather than papered over.

**A Council Secretary discrepancy, resolved by dating the evidence rather than picking one name.** Two different individuals are documented as Kalomo's Council Secretary in different sources: Lisa Mpasela (2023 IDP launch, and a CDF-project news post referencing the 2023 funding phase) and Trophius Kufanga (the council's Web-Based MIS launch article — undated, but referencing 2026-era infrastructure). Cross-referencing found that a Trophius Kufanga was separately Council Secretary at **Masaiti** Town Council per a dated 2023 public notice — consistent with the routine practice of council secretaries transferring between local authorities. Rather than guessing which name is "current", both are recorded as rows with the supporting context and the caveat that the exact handover date isn't stated in either source.

In [8]:
raw_dir = os.path.join("..", "data", "raw", "administrative_governance")
notes_path = os.path.join(raw_dir, "source_pages_extract.txt")
with open(notes_path, encoding="utf-8") as f:
    notes = f.read()
print(f"{notes_path}: {len(notes)} chars of compiled source notes")
print()
print(notes[:1200], "...")

..\data\raw\administrative_governance\source_pages_extract.txt: 21062 chars of compiled source notes

SOURCE PAGE EXTRACTS - Kalomo Town Council administrative/governance dataset
Collected by: Nicholas
Method: scripts/scraping/scrape_administrative_governance.py targets these
exact pages on https://www.kalomocouncil.gov.zm/. The council site blocks
automated/agent HTTP clients (returns 403 to a plain requests.get from this
environment, and is robots-disallowed for hosted fetch tools), so the
running notes below were compiled by cross-referencing indexed copies of
each page's rendered text (the same pages the script targets) together with
the council's own news posts, which are separately indexed and were fully
readable. This mirrors the approach already used for the scanned/rotated
PDFs in data/raw/development_plans (documented from listing/context rather
than full text) - content is recorded honestly per source, with gaps noted
rather than invented. Running the script from a normal re

## Step 4 — Structuring the notes into rows

As with the development-plans dataset, this content is prose, not tables, so `source_pages_extract.txt` was read by hand and each governance fact pulled out into `clean_administrative_governance.py`. Six row categories were used:

1. **Leadership** — Council Chairperson, Council Secretary (both documented, see Step 3), District Commissioner (flagged as a central-government, not Council, office), Directors of Finance/Engineering/ICT, and the district's two Members of Parliament (flagged as national, not Council, offices — included because the council's own site organises its Civic Leaders page around these two constituencies).
2. **Department** — the four departments/offices directly confirmed by name in council sources (Council Secretary's office, Finance, Engineering, ICT). Departments typical of comparable Zambian councils (Public Health, Planning, Human Resource) were deliberately **not** added without direct confirmation.
3. **Ward** — the one ward councillor confirmed via indexed content.
4. **Committee** — the grassroots governance bodies named together in the council's own R-CFW sensitization news post: Ward Development Committees, Community Welfare Assistance Committees, Satellite Disaster Management Committees, and Headmen, each present across all 20 wards.
5. **Constituency** — Kalomo Central and Dundumwezi.
6. **Chiefdom** — Chikanta, Siachitema, Sipatunyana.

`record_type` extends the illustrative list in `docs/DATA_DICTIONARY.md` (which was always marked "e.g.") to match what the sources actually supported, and the dictionary was updated to document the change and the coverage caveat.

**Standardisation applied**, matching the conventions agreed for every dataset in this project:
- `record_id` assigned sequentially as `AG-001` … `AG-023`
- Every text field stripped of whitespace; empty/`nan`/`None` normalised to `"N/A"`
- Duplicate rows dropped on `(name_or_title, source_url)`
- Every row asserted to have a working `https://` `source_url` and a unique `record_id`
- `date_scraped` stamped with the date the cleaning script was run
- Output written with `sep="|"` per the project's naming/format convention

## Step 5 — Loading and inspecting the final dataset

In [9]:
import pandas as pd

csv_path = os.path.join("..", "data", "processed", "db-unza26-csc4792-kalomo_town_council_administrative_governance.csv")
df = pd.read_csv(csv_path, sep="|", keep_default_na=False)
df.head(10)

,record_id,record_type,name_or_title,role_or_function,ward,date,source_url,date_scraped
0,AG-001,Contact,Kalomo Town Council - General Contact Information,Email: towncouncilkalomo@gmail.com | Address: ...,N/A,N/A,https://www.kalomocouncil.gov.zm/?page_id=770,2026-09-12
1,AG-002,Leadership,Coy Makaya,Council Chairperson,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
2,AG-003,Leadership,Lisa Mpasela,Council Secretary (per the 2023 IDP launch and...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=1799,2026-09-12
3,AG-004,Leadership,Trophius Kufanga,Council Secretary (per the council's Web-Based...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
4,AG-005,Leadership,Joshua Munsaka Sikaduli,District Commissioner (Office of the President...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
5,AG-006,Leadership,Jimmy Mubanga,Director of Finance,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
6,AG-007,Leadership,Joel Mweempe,Director of Engineering (per the 2026 budget c...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
7,AG-008,Leadership,Judith Beene,Director of Information and Communication Tech...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
8,AG-009,Leadership,Harry Kamboni,"Member of Parliament, Kalomo Central Constitue...",N/A,N/A,https://en.wikipedia.org/wiki/Kalomo_Central,2026-09-12
9,AG-010,Leadership,Valencia Simwale,Vice Council Chairperson; also serves as the N...,Naluja Ward,N/A,https://www.kalomocouncil.gov.zm/?page_id=2871,2026-09-12


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   record_id         88 non-null     str  
 1   record_type       88 non-null     str  
 2   name_or_title     88 non-null     str  
 3   role_or_function  88 non-null     str  
 4   ward              88 non-null     str  
 5   date              88 non-null     str  
 6   source_url        88 non-null     str  
 7   date_scraped      88 non-null     str  
dtypes: str(8)
memory usage: 5.6 KB


In [11]:
print("Rows by record_type:")
print(df["record_type"].value_counts())
print("\nRows with ward = N/A:", (df["ward"] == "N/A").sum(), "out of", len(df))
print("Rows with date = N/A:", (df["date"] == "N/A").sum(), "out of", len(df))

Rows by record_type:
record_type
Leadership      24
Committee       21
Ward            20
Department       9
Resolution       6
Chiefdom         3
Constituency     2
Contact          1
Service          1
Report           1
Name: count, dtype: int64

Rows with ward = N/A: 45 out of 88
Rows with date = N/A: 46 out of 88


## Step 6 — Closing the resolutions/reports gap with uploaded primary sources

Issue #3 asked for "meeting resolutions, public notices, and reports" as one of three data-point categories. Through Step 5, none of this had been found on the public website — searches for tenders, notices, and minutes on `kalomocouncil.gov.zm` came up empty.

Nicholas then supplied three scanned council documents directly:

1. Dundumwezi Constituency Development Fund Committee minutes, 29 December 2023
2. Kalomo Central Constituency Development Fund Committee minutes, 9-10 February 2024
3. Community Engagement Meeting on Budget Preparation minutes, 28 November 2024

These have no public URL, so `source_url` for the rows they produced cites the document title and date instead (clearly labelled `"uploaded scan, no public URL"` so it's never confused with a live web page).

**What was extracted, and what wasn't:** these documents are dense — the Kalomo Central minutes alone carry a 33-line CDF project table, and the Nov 2024 minutes carry multi-page financial performance tables and a 300+ row CDF project-approval appendix. None of that project- or finance-level detail was copied into *this* dataset — it belongs to the team's separate `cdf_projects` and `financial_data` datasets (both exist as their own files in this repo, owned by other team members), and duplicating it here would blur the line between datasets. What *was* extracted, because it's squarely governance data:

- The two CDF Committees themselves (membership, chairperson, MP) — new `Committee` rows
- Named WDC Chairpersons for 14 of the district's 20 wards, from the Nov 2024 meeting's attendance list — a major upgrade over the single generic "WDC present in every ward" row from Step 4
- Several council officers not previously in the dataset (Chief Accountant, Internal Auditor, Council Advocate, Valuation Officer, Committee Clerks, and others) — new `Leadership` rows
- Six governance-level resolutions (minutes adoption, an empowerment-loan ward-allocation formula, application approvals, a ZDSP project list) — the dataset's first `Resolution` rows
- The fact that a budget-performance report was presented, without the figures themselves — the dataset's first `Report` row

Two minor discrepancies surfaced across documents and are recorded rather than silently resolved: Joel Mweempe's title/spelling varies ("Director of Engineering" vs. "Joel Mwempe, Ass. Director of Engineering Services"), and Christopher Zyambo is titled "District Treasurer" in one document and "Council Treasurer" in another — same people, kept as one row each with both variants noted.

In [12]:
df = pd.read_csv(csv_path, sep="|", keep_default_na=False)
print("Updated record_type breakdown:")
print(df["record_type"].value_counts())
print()
print("Rows citing an uploaded document (no public URL):",
      df["source_url"].str.contains("uploaded scan").sum())
print("Named WDC Chairperson rows:",
      df["name_or_title"].str.contains("WDC\\) Chairperson").sum())

Updated record_type breakdown:
record_type
Leadership      24
Committee       21
Ward            20
Department       9
Resolution       6
Chiefdom         3
Constituency     2
Contact          1
Service          1
Report           1
Name: count, dtype: int64

Rows citing an uploaded document (no public URL): 38
Named WDC Chairperson rows: 14


## Limitations

- Ward councillor roster is now complete (20/20 wards, manually captured directly from the live Civic Leaders page on 2026-09-11 after indexed retrieval fell short — see Step 3).
- No confirmed department directory beyond the 4 departments directly named in council sources; other departments typical of comparable councils were deliberately excluded rather than assumed.
- Most `date` values remain `"N/A"` — the council's news posts and pages mostly omit explicit publish dates in their indexed text. A few dates were recovered where the source's own dated feed was visible (e.g. the Institutional Management page's activity feed confirmed 16 June 2026 for the R-CFW sensitization post, and a separate news post confirmed 11 September 2024 for the Food Security Pack Program).
- The Council Secretary discrepancy (Lisa Mpasela vs. Trophius Kufanga) is presented as two dated-as-best-as-possible rows with explicit caveats rather than resolved to a single "current" answer, since neither source states an exact handover date - though the uploaded Nov 2024 minutes do explicitly confirm Trophius Kufanga as Council Secretary and meeting chairperson as of that date.
- The named WDC Chairperson roster is 14/20 wards, not 20/20 - the remaining 6 wards' chairpersons weren't named in the one attendance list available. The generic "WDC present in every ward" Committee row from Step 4 still covers all 20.
- Resolution and Report rows are a curated subset of what the uploaded minutes contain, not a full transcription - detailed CDF project tables and financial performance figures were deliberately left to the team's cdf_projects and financial_data datasets (see Step 6).
- Members of Parliament are included for completeness of the district's civic-leadership picture (the council's own Civic Leaders page is organised by constituency) but are explicitly flagged as national, not Council, offices.

See `docs/DATA_DICTIONARY.md` for the full column reference for this and every other dataset in this project, including the coverage note added for this dataset.

---

# Part 3 — Financial & Revenue Data (Faiz)

Source notebook: `notebooks/financial_data_notebook.ipynb`


# Kalomo Town Council — Financial & Revenue Data
### CSC4792 Mini Project — Financial/Revenue component (Faiz)

This notebook documents how the financial and revenue dataset for
**Kalomo Town Council** was scraped, extracted, and cleaned from the
council's official website (kalomocouncil.gov.zm), as required by the
CSC4792 mini-project brief (Issue #2: "Faiz: Scrape financial and
revenue data").

**What this dataset covers** (per the assignment requirements and
Issue #2's data points):
- Approved budgets (by fiscal year)
- Own Source Revenue (OSR) targets and actual collections
- Central Government transfers / cooperating partner funding
- Local Government Equalisation Fund (LGEF) allocations
- Actual expenditure vs. approved budget
- Local revenue sources: market fees, local taxes, fees and charges
- Budget performance percentages (`percent_of_target`)

**A note on method:** the council's `robots.txt` disallows automated
crawling. The scraping scripts here use plain `requests`, which does not
enforce robots.txt automatically, but this is disclosed here and in the
Data in Brief paper for transparency, and requests are rate-limited
(`time.sleep`) to avoid hammering the server.


## Step 1: Identify source pages

Financial information on the council's site is published as ordinary
news posts (WordPress `?p=NNNN` URLs) and as downloadable PDFs (budget
documents, investment profiles, CDF newsletters). These were located by
manually browsing the site and are listed in
`data/raw/financial_data/urls.txt`.

In [13]:
import os
path = "../data/raw/financial_data/urls.txt"
if os.path.exists(path):
    with open(path) as f:
        print(f.read())
else:
    print(f"NOTE: {path} was not committed to the repo (only the notebook was) - flagged on issue #2 for Faiz to push it.")


NOTE: ../data/raw/financial_data/urls.txt was not committed to the repo (only the notebook was) - flagged on issue #2 for Faiz to push it.


## Step 2: Scrape the HTML pages and PDFs

Three scripts do the scraping:
- `scripts/scraping/scrape_financial_html.py` — downloads each HTML news
  page and extracts the title and main article text using BeautifulSoup.
- `scripts/scraping/scrape_financial_pdfs.py` — downloads each linked
  PDF and extracts its text using `pdfplumber`.
- `scripts/scraping/crawl_and_scrape_financial.py` — a broader crawler
  that follows internal links from the seed URLs in `urls.txt` to find
  additional budget/financial pages and PDFs that aren't linked directly
  from the pages we started with (e.g. `2025-Revised-Budget.pdf`,
  `2026-BUDGET-CONSULTATIVE-MEETING.pdf`). Its progress is checkpointed
  in `data/raw/financial_data/_crawl_state.json` so it can resume without
  re-downloading pages, and every URL it visits is logged in
  `all_urls_visited.txt` for full provenance.

All scraping saves its raw output into `data/raw/financial_data/` as CSV
files (`raw_scraped_articles.csv` and `raw_scraped_pdfs.csv`) and the
raw PDFs themselves are kept in `data/raw/financial_data/pdfs/`. This raw
stage deliberately keeps the full article/PDF text rather than numbers,
so nothing is lost before the extraction step.


In [14]:
import os, pandas as pd
path = "../data/raw/financial_data/raw_scraped_articles.csv"
if os.path.exists(path):
    raw_articles = pd.read_csv(path)
    display(raw_articles[["source_url", "title"]])
else:
    print(f"NOTE: {path} was not committed to the repo (only the notebook was) - flagged on issue #2 for Faiz to push it.")


NOTE: ../data/raw/financial_data/raw_scraped_articles.csv was not committed to the repo (only the notebook was) - flagged on issue #2 for Faiz to push it.


## Step 3: Extract structured financial figures

Two different extraction strategies are needed because council figures
appear in two very different formats in the raw text:

**1. Narrative figures** — plain sentences in news posts/meeting minutes, e.g.:

> "Kalomo Town Council adopted a total approved budget of K131,360,483
> for the 2026 financial year..."

`extract_narrative_records()` in `scripts/cleaning/clean_financial_data.py`
uses targeted regular expressions to pull out each K-amount and classify
it (e.g. `total_approved_budget`, `expenditure_actual`) based on the
surrounding wording.

**2. Table figures** — budget PDFs where `pdfplumber` flattens a table
into a single line of text with no column boundaries, e.g.:

```
Local Government Equalisation Fund 11,706,441 ...
Market fees 250,000 ...
Local Taxes 1,407,959 703,612 47
```

`extract_table_records()` handles these with its own set of patterns
(LGEF, market fees, local taxes, fees and charges), pulling the fiscal
year from the source filename/URL when it isn't stated in the row
itself. These table-based rules are noted as more fragile in the data
dictionary, since they depend on the exact column order the council
used in that particular document.

Both strategies are explicit, readable rules rather than generic NLP,
so every extracted number can be traced back to exactly why it was
classified that way — important for a small, council-specific dataset
where accuracy matters more than scale.

**A bug worth documenting:** an early version of the expenditure rule
silently failed to match because `pdfplumber` had inserted a line break
in the middle of the sentence ("total expenditure for\nthe period..."),
and a literal space in a regex does not match a newline. The fix,
applied in `extract_records_from_text()`, is to collapse all whitespace
(spaces, tabs, newlines) in the scraped text down to single spaces
*before* running any extraction rule. This is a good general lesson for
PDF-derived text: never assume a sentence stays on one line.


## Step 4: Clean and structure the data

Cleaning steps applied:
- Collapse whitespace in raw scraped text before extraction (see the bug
  note in Step 3) so line-wrapped sentences don't break the regex rules.
- Strip the `K` currency symbol and thousands separators, convert to
  numeric (float) values in ZMW.
- Classify each figure as a `target` (budgeted/projected) or `actual`
  (collected/spent) value.
- Compute `percent_of_target` automatically where both a target and a
  matching actual figure exist for the same fiscal year — this covers
  OSR, Central Government transfers, local taxes, fees and charges, and
  expenditure vs. approved budget.
- Remove exact duplicate records (same figure appearing on more than one
  page).
- Add a `notes` field for figures with important context (e.g. the OSR
  drop attributed to lower plot premium projections, or a flag that a
  figure came from a fragile flattened PDF table).


In [15]:
import subprocess
subprocess.run(["python3", "../scripts/cleaning/clean_financial_data.py"])

CompletedProcess(args=['python3', '../scripts/cleaning/clean_financial_data.py'], returncode=9009)

## Step 5: Final dataset preview

In [16]:
df = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kalomo_town_council_financial_data.csv",
    sep="|"
)
df

,council_name,fiscal_year,record_type,amount_zmw,target_or_actual,source_url,scrape_date,percent_of_target,notes
0,Kalomo Town Council,2024,expenditure_actual,77076699.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,80.8,NaN
1,Kalomo Town Council,2024,total_approved_budget,95399253.0,target,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,NaN,NaN
2,Kalomo Town Council,2025,central_govt_transfers_budget,135701875.0,target,https://www.kalomocouncil.gov.zm/?p=4687,2026-09-12,NaN,NaN
3,Kalomo Town Council,2025,fees_and_charges_actual,1260217.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,68.0,Extracted from a flattened PDF budget table; v...
4,Kalomo Town Council,2025,fees_and_charges_actual,210000.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,11.3,Extracted from a flattened PDF budget table; v...
5,Kalomo Town Council,2025,fees_and_charges_actual,200000.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,10.8,Extracted from a flattened PDF budget table; v...
6,Kalomo Town Council,2025,fees_and_charges_budget,1854513.0,target,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,NaN,Extracted from a flattened PDF budget table; v...
7,Kalomo Town Council,2025,fees_and_charges_budget,200000.0,target,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,NaN,Extracted from a flattened PDF budget table; v...
8,Kalomo Town Council,2025,lgef_disbursement,11706441.0,target,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,NaN,Extracted from a flattened PDF budget table; v...
9,Kalomo Town Council,2025,local_taxes_actual,713855.0,actual,http://www.kalomocouncil.gov.zm/wp-content/upl...,2026-09-12,47.8,Extracted from a flattened PDF budget table; v...


## Step 6: Output format compliance check

Per the assignment brief and Issue #2's acceptance criteria, the final
CSV must:
- Be in CSV format ✅
- Use `|` as the column separator ✅ (`sep="|"` used above and in export)
- Follow the naming convention `db-unza26-csc4792-[description].csv` ✅
  → `db-unza26-csc4792-kalomo_town_council_financial_data.csv`
- Have numeric amounts with no stray currency symbols/commas ✅
  (`to_number()` strips `K`, commas, and spaces before converting to float)
- Record a `source_url` and `scrape_date` for every row ✅
- Be reflected in `docs/DATA_DICTIONARY.md` ✅


## Next steps / limitations

This notebook now covers approved budgets, LGEF, expenditure, and local
revenue (market fees, local taxes, fees and charges) across fiscal years
2024–2026, sourced from both prose (news posts, meeting minutes) and
flattened PDF budget tables.

Known limitations to flag in the Data in Brief paper:
- Table-based rows (`_budget`/`_actual` figures pulled from flattened
  PDF tables) are more fragile than prose-based rows — always spot-check
  a few against the source PDF listed in `source_url`.
- Some fiscal years only have a `target` or only an `actual` figure for
  a given record type (no matching pair yet), so `percent_of_target` is
  blank for those rows — this is expected, not a bug.
- To make the dataset richer, extend `urls.txt` / let the crawler run
  further to pick up additional quarterly budget performance reports,
  older years' budgets, and CDF-specific disbursement reports, then add
  matching extraction rules to `clean_financial_data.py` for any new
  phrasing patterns found.


---

# Part 4 — Constituency Development Fund (CDF) Projects (Louis)

Source notebook: `notebooks/cdf_projects.ipynb`


# CDF and Community Projects Dataset - Kalomo Town Council

**Owner:** Louis | **CSC4792 Mini Project, Group 44 - Kalomo Town Council, Zambia**

This notebook documents how `db-unza26-csc4792-kalomo_town_council_cdf_projects.csv` was produced from scanned Constituency Development Fund (CDF) project lists published by Kalomo Town Council. It covers source selection, OCR extraction, table reconstruction, cleaning decisions, validation, and the limitations of the final dataset.

The supporting scripts are:

- `scripts/scraping/scrape_cdf_projects.py`
- `scripts/cleaning/clean_cdf_projects.py`

The expensive OCR stage was run in Google Colab because EasyOCR benefits from a GPU. The notebook uses the saved raw and processed CSV files, so it can be reviewed without downloading the PDFs or running OCR again.

## Step 1 - Identify and select the source documents

The council website provided 20 CDF-related PDFs. They were reviewed before scraping because they did not all describe the same type of record. Some contained community projects, while others contained individual skills-development or boarding-school bursary beneficiaries.

Four PDFs were selected for this dataset:

1. `CDF-DUNDUMWEZI-2024.pdf`
2. `CDF-KALOMO-CENTRAL-2024.pdf`
3. `2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJECTS.pdf`
4. `2025-KALOMO-CENTRAL-NOT-APPROVED-AND-APPROVED-PROJECTS.pdf`

These are the complete project lists for the two constituencies and two years. Separate approved-only and not-approved-only PDFs were not added because their records are already contained in the combined lists. Bursary registers were also excluded because their rows represent people rather than projects and contain personal identifiers that do not belong in the agreed CDF project schema.

In [17]:
from pathlib import Path
import sys

repo_root = Path("..").resolve()
if not (repo_root / "scripts").exists():
    repo_root = Path(".").resolve()
sys.path.insert(0, str(repo_root / "scripts" / "scraping"))

import scrape_cdf_projects as scraper

print("Canonical project PDFs used:")
for filename in scraper.CANONICAL_PROJECT_PDFS:
    print(" -", filename)

Canonical project PDFs used:
 - CDF-DUNDUMWEZI-2024.pdf
 - CDF-KALOMO-CENTRAL-2024.pdf
 - 2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJECTS.pdf
 - 2025-KALOMO-CENTRAL-NOT-APPROVED-AND-APPROVED-PROJECTS.pdf


## Step 2 - Extract the scanned tables with OCR

The selected PDFs are scanned documents, so ordinary PDF text extraction does not recover their tables reliably. `scrape_cdf_projects.py` renders each page at 300 DPI with `pypdfium2` and sends the page image to EasyOCR.

EasyOCR returns the detected text, a confidence score, and the coordinates of its bounding box. The coordinates are important because OCR detects individual table cells and text fragments, not complete project records. The scraper therefore saves the following raw fields:

- `text_line`
- `source_file_name`
- `page_number`
- `x_min`, `y_min`, `x_max`, `y_max`
- `ocr_confidence`

The OCR command used in Colab was:

```bash
python scripts/scraping/scrape_cdf_projects.py --pdf-dir /content/cdf_pdfs --gpu
```

The result was 3,804 OCR fragments saved to `data/raw/cdf_projects/raw_cdf_projects.csv`. A fragment is only part of a table row, so this number is not the number of projects.

In [18]:
import pandas as pd

raw_path = repo_root / "data" / "raw" / "cdf_projects" / "raw_cdf_projects.csv"
raw = pd.read_csv(raw_path)

print("Raw OCR fragments:", len(raw))
print("Source PDFs:", raw["source_file_name"].nunique())
display(raw.head())
display(raw.groupby(["source_file_name", "page_number"]).size().rename("fragments").to_frame())

Raw OCR fragments: 3804
Source PDFs: 4


,text_line,source_file_name,page_number,x_min,y_min,x_max,y_max,ocr_confidence
0,MINISTRY OF LOCAL GOVERNMENT AND RURAL DEVELOF...,2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJ...,1,1517.0,197.0,2498.0,233.0,0.55951
1,DUNDUMWEZI COMMUNITY PROJECT- 2025,2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJ...,1,1818.0,235.0,2477.0,271.0,0.90752
2,Reason,2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJ...,1,2909.0,276.0,3011.0,312.0,0.99993
3,Project Description,2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJ...,1,1390.0,278.0,1627.0,317.0,0.87476
4,Type of Project,2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJ...,1,2180.0,279.0,2372.0,315.0,0.73170


fragments
source_file_name                                   page_number           
2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJE... 1                  383
                                                   2                  410
                                                   3                  417
                                                   4                  420
                                                   5                  427
                                                   6                  411
                                                   7                  338
2025-KALOMO-CENTRAL-NOT-APPROVED-AND-APPROVED-P... 1                  380
                                                   2                  401
CDF-DUNDUMWEZI-2024.pdf                            1                   76
CDF-KALOMO-CENTRAL-2024.pdf                        1                  141

## Step 3 - Problems encountered

**The first OCR file was much larger than expected.** Running OCR across all 20 PDFs produced about 28,000 rows. This happened because one OCR text box was written as one CSV row and because the run included thousands of bursary beneficiary entries. It did not mean that the council had 28,000 projects.

**Some project documents overlap.** The site publishes complete combined lists as well as separate approved and not-approved versions. Treating all of them as independent sources would count the same applications more than once. The scraper was restricted to the four complete sources listed in Step 1.

**Reading OCR fragments in sequence merged table rows.** An early cleaner treated the OCR output as plain text. This produced short or mixed names such as ward names, descriptions, and headings being mistaken for projects. The corrected scraper retains bounding-box coordinates, and the cleaner uses those coordinates to rebuild the original table columns and rows.

**OCR spelling is not always exact.** Examples include `NOL APPROVED` for `NOT APPROVED`, `AIl Wards` for `All Wards`, and several misspellings of `Infrastructure`. The cleaner uses a small list of expected wards and sectors together with conservative fuzzy matching. It leaves a field as `N/A` when the source text is still too unclear to classify safely.

## Step 4 - Reconstruct and clean the project records

The 2024 and 2025 PDFs use different table layouts, so the cleaning script handles them separately.

For the 2024 tables, each recognised sector cell marks one project row. The bottom edge of the previous sector box and the current sector box are used to estimate the row boundaries. Text in the project-name and ward columns is then collected from the same vertical area.

For the 2025 tables, each `Approved` or `Not Approved` cell marks a project application. Repeated table headings are used to locate the project-name, description, sector, type, ward, comments, and reason columns. Nearby serial numbers provide an extra boundary when project names are tightly packed.

The cleaner also:

- removes headings, serial numbers, rejection reasons, stamps, and letterhead text from project names;
- standardises recognised ward, sector, and status values;
- keeps legitimate repeated applications instead of dropping them automatically;
- assigns sequential IDs from `CDF-0001`;
- records the council source page and cleaning date;
- checks project names for NRC and phone-number patterns before export; and
- writes the final file with a pipe (`|`) delimiter.

The published project tables do not provide reliable allocation or disbursement amounts for these rows, so `amount_allocated_zmw` and `amount_disbursed_zmw` are recorded as `N/A` rather than guessed.

In [19]:
sys.path.insert(0, str(repo_root / "scripts" / "cleaning"))
import clean_cdf_projects as cleaner

print("Final column order:")
for column in cleaner.OUTPUT_COLUMNS:
    print(" -", column)

print("\nTo rebuild the processed file without repeating OCR:")
print("python scripts/cleaning/clean_cdf_projects.py")

Final column order:
 - project_id
 - project_name
 - ward
 - sector
 - amount_allocated_zmw
 - amount_disbursed_zmw
 - fiscal_year
 - status
 - source_url
 - date_scraped

To rebuild the processed file without repeating OCR:
python scripts/cleaning/clean_cdf_projects.py


## Step 5 - Load and inspect the final dataset

In [20]:
processed_path = repo_root / "data" / "processed" / "db-unza26-csc4792-kalomo_town_council_cdf_projects.csv"
df = pd.read_csv(
    processed_path, sep="|", keep_default_na=False, dtype={"fiscal_year": str}
)
df.head(10)

,project_id,project_name,ward,sector,amount_allocated_zmw,amount_disbursed_zmw,fiscal_year,status,source_url,date_scraped
0,CDF-0001,Fabrication And Delivery Of 2500 Double Sitter...,All Wards,Education,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
1,CDF-0002,Ambulance,All Wards,Health,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
2,CDF-0003,Motor Bike-Chikanta Chiefdom,All Wards,Traditional Affairs,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
3,CDF-0004,Motor Bike-Siachitema Chiefdom,All Wards,Traditional Affairs,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
4,CDF-0005,Chiefs Palace-Chikanta,All Wards,Traditional Affairs,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
5,CDF-0006,Chiefs Palace-Siachitema,All Wards,Traditional Affairs,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
6,CDF-0007,Construction Of Police Post In Kasukwe,All Wards,Home Affairs,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
7,CDF-0008,Construction Mortuary At Chikanta Habulile Min...,Chikanta,Health,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
8,CDF-0009,Lubricants For Earth Moving Equipment,All Wards,Road Infrastructure,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12
9,CDF-0010,Rural Electrification (Rea),All Wards,Energy,N/A,N/A,2024,Approved,https://www.kalomocouncil.gov.zm/?page_id=3025,2026-09-12


In [21]:
print("Total project records:", len(df))
print("\nRecords by year and status:")
display(df.groupby(["fiscal_year", "status"]).size().rename("records").to_frame())

print("Records by sector:")
display(df["sector"].value_counts().rename("records").to_frame())

print("Records by ward:")
display(df["ward"].value_counts().rename("records").to_frame())

Total project records: 488

Records by year and status:


records
fiscal_year status               
2024        Approved           53
2025        Approved           74
            Not Approved      361

Records by sector:


,records
sector,
Water and Sanitation,211
Education,162
Health,49
Livestock,17
Infrastructure,14
Traditional Affairs,8
Agriculture,6
Road Infrastructure,4
Energy,4


Records by ward:


,records
ward,
Katanda,70
Chamuka,69
Bbilili,48
Chikanta,45
Omba,45
Naluja,30
All Wards,21
Mikata,20
Kasukwe,17


## Step 6 - Validate the processed file

The checks below confirm the agreed schema, unique project IDs, recognised years and statuses, and the absence of personal identifier patterns. The expected result is 488 records: 53 from 2024 and 435 from 2025.

In [22]:
import re

expected_columns = list(cleaner.OUTPUT_COLUMNS)
assert list(df.columns) == expected_columns
assert len(df) == 488
assert df["project_id"].is_unique
assert (df["project_name"] != "N/A").all()
assert set(df["fiscal_year"]) == {"2024", "2025"}
assert set(df["status"]) <= {"Approved", "Not Approved"}

nrc_pattern = re.compile(r"\b\d{4,7}\s*/\s*\d{1,3}(?:\s*/\s*\d)?\b")
phone_pattern = re.compile(r"\b0?9[567]\d{7}\b")
search_text = df.astype(str).agg(" ".join, axis=1)
assert not search_text.str.contains(nrc_pattern, regex=True).any()
assert not search_text.str.contains(phone_pattern, regex=True).any()

print("All validation checks passed.")
print("Missing ward values:", (df["ward"] == "N/A").sum())
print("Missing sector values:", (df["sector"] == "N/A").sum())

All validation checks passed.
Missing ward values: 1
Missing sector values: 1


## Step 7 - Output format compliance

The final dataset follows the repository conventions:

- Filename: `db-unza26-csc4792-kalomo_town_council_cdf_projects.csv`
- Location: `data/processed/`
- Delimiter: pipe (`|`)
- Schema: the 10 CDF project columns documented in `docs/DATA_DICTIONARY.md`
- Unique identifier: `project_id`
- Provenance: `source_url` and `date_scraped` on every row

The raw OCR file remains under `data/raw/cdf_projects/` so the cleaning stage is reproducible without keeping the temporary PDF folder in the repository.

## Limitations

- OCR spelling errors remain in a small number of project names because the original PDFs are scanned images. Values were not manually rewritten unless the intended label could be matched confidently.
- One 2024 disaster-component record covers several health posts and does not identify one clear ward, so its ward is `N/A`.
- One 2025 sector cell remains unreadable after OCR and is kept as `N/A` rather than guessed.
- Allocation and disbursement amounts are `N/A` because the selected project tables do not provide reliable project-level figures in those columns.
- The dataset covers community projects only. Individual bursary beneficiaries were deliberately excluded because they require a different schema and introduce unnecessary personal information.
- The temporary source PDFs are not committed to the repository. The source URLs identify the council pages from which the documents were obtained.

See `docs/DATA_DICTIONARY.md` for the complete column descriptions.

---

# Cross-dataset summary

Quick preview of every dataset in one place — all four datasets are now complete.

In [23]:
import pandas as pd
import os

datasets = {
    "Development plans": "db-unza26-csc4792-kalomo_town_council_development_plans.csv",
    "Administrative & governance": "db-unza26-csc4792-kalomo_town_council_administrative_governance.csv",
    "Financial & revenue": "db-unza26-csc4792-kalomo_town_council_financial_data.csv",
    "CDF projects": "db-unza26-csc4792-kalomo_town_council_cdf_projects.csv",
}

summary = []
for name, fname in datasets.items():
    path = os.path.join("..", "data", "processed", fname)
    df = pd.read_csv(path, sep="|", keep_default_na=False)
    summary.append({"dataset": name, "rows": len(df), "columns": len(df.columns), "column_names": ", ".join(df.columns)})

pd.DataFrame(summary)

,dataset,rows,columns,column_names
0,Development plans,20,9,"record_id, plan_name, plan_type, sector, perio..."
1,Administrative & governance,88,8,"record_id, record_type, name_or_title, role_or..."
2,Financial & revenue,18,9,"council_name, fiscal_year, record_type, amount..."
3,CDF projects,488,10,"project_id, project_name, ward, sector, amount..."
